In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key not set (and this is optional)
Grok API Key not set (and this is optional)


In [3]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)


In [4]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8250U CPU @ 1.60GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Short answer: you don’t have a C++ compiler installed right now. You’ll need to install one.

Simplest way on your system (Windows 11 with winget): install the “Visual Studio Build Tools” C++ toolchain.

Steps
1) Open Windows Terminal (PowerShell) as Administrator.
2) Run this command to install the MSVC C++ build tools and the Windows SDK:
   winget install -e --id Microsoft.VisualStudio.2022.BuildTools --override "--add Microsoft.VisualStudio.Workload.VCTools --includeRecommended --passive --norestart"
3) When the installer finishes, you’re set. No reboot is typically required; if cl isn’t found later, sign out/in or open a new terminal.

Python commands to compile and run main.cpp
Use MSVC with high-performance flags. The compile command below locates your VS installation with vswhere, sets up the environment, then compiles and links:

compile_command = [
    "cmd", "/d", "/s", "/c",
    r'for /f "usebackq tokens=*" %i in (`"%ProgramFiles(x86)%\Microsoft Visual Studio\Installer\vswhere.exe" -latest -products * -requires Microsoft.VisualStudio.Component.VC.Tools.x86.x64 -property installationPath`) do (call "%i\VC\Auxiliary\Build\vcvars64.bat" && cl /O2 /GL /std:c++20 /EHsc /MD /arch:AVX2 /fp:fast /DNDEBUG main.cpp /Fe:main.exe /link /LTCG)'
]

run_command = ["main.exe"]

Notes
- The flags aim for fast runtime: /O2, /GL + /LTCG (link-time optimization), /fp:fast, /DNDEBUG. /arch:AVX2 targets your CPU for extra speed; if you need broader compatibility on other machines, remove /arch:AVX2.
- If you prefer, you can instead run your Python script from the “x64 Native Tools Command Prompt for VS 2022,” in which case compile_command can simply start with ["cl", ...] and you don’t need the cmd/vcvars64 wrapper.

In [7]:
import subprocess
import os

work_dir = r"C:\Users\prate\Repositories\AI\AI_Study\AI_Code_Converter"
vsdevcmd = r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\Common7\Tools\VsDevCmd.bat"

# Build the inner command as a single string for shell=True
compile_command = (
    f'"{vsdevcmd}" -arch=x64 && '
    f'cd /d "{work_dir}" && '
    'cl /nologo /std:c++20 /O2 /GL /arch:AVX2 /fp:fast /EHsc /DNDEBUG main.cpp /Fe:main.exe /link /LTCG'
)

run_command = os.path.join(work_dir, "main.exe")

# --- compile ---
print("Compiling...")
result = subprocess.run(compile_command, capture_output=True, text=True, shell=True, cwd=work_dir)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Compilation failed")
print("Compilation successful!")

# --- run ---
print("Running:")
print(subprocess.run(run_command, capture_output=True, text=True).stdout)


Compiling...
**********************************************************************
** Visual Studio 2022 Developer Command Prompt v17.0
** Copyright (c) 2025 Microsoft Corporation
**********************************************************************
main.cpp
Generating code
Finished generating code

Compilation successful!
Running:
Hello from C++!



In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [10]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [11]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""


In [13]:

def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [14]:
run_python(pi)


Result: 3.141592656089
Execution Time: 57.833880 seconds


In [15]:
port(openai, OPENAI_MODEL, pi)

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()